# G12: G9's eval_mini on OLMoE GGUF files (Kaggle or Colab)

Runs G9's **mini** profile (ARC-C 500, HellaSwag 400, MMLU 570, GSM8K 200, TruthfulQA-mc2 200; G9's shots, limits
and seed) on **bf16 and six llama.cpp formats**, all through one backend (llama-cpp-python in-process; see
`llamacpp_lm.py` for why not llama-server), with **per-item logs**, then compares each format with bf16 item by
item (`paired.py`).

**Team run (G12):** bf16 is done (Kaggle, 28 Oct). Each member runs the formats assigned to them: set `FORMATS` and
`RUN_BY` in the settings cell, nothing else. Send the zip back; `combine.py` checks that every run used identical code
and settings (the `NOTEBOOK_VERSION` fingerprint, model revision, llama.cpp build, lm_eval, seed, item counts) and
compares every format with bf16 item by item.

Every step stops the run if it fails. The notebook detects where it runs:

- **Kaggle:** Settings → Accelerator **GPU T4 x2**, Internet **on**. For the full run use **Save Version → Save & Run
  All (Commit)**: it runs in the background (up to 12 h) and `/kaggle/working/eval_mini_out/` becomes the version's
  downloadable Output. Model files go to `/kaggle/tmp` (outside the 20 GB output limit).
- **Colab:** T4 runtime, Run all. Output to Google Drive if it mounts (then a second session reuses the compiled
  build and skips finished tasks), else to the runtime, with the zip also printed by the last cell.
- Time on Kaggle 2×T4: ~35 min to compile llama-cpp-python and convert the model, then ~25-45 min per format
  (bf16 took 45 min). Two formats ≈ 2 h. `LIMIT = 5` gives a quick end-to-end check first if you want one.

In [ ]:
# --- settings -----------------------------------------------------------------------------------------------
FORMATS   = ["Q8_0", "Q6_K"]   # <-- YOUR assigned formats (bf16 is already done; see the team table in README.md)
RUN_BY    = ""                 # <-- your GitHub handle, recorded with the results
LIMIT     = None               # None = G9's full mini profile; 5 = quick end-to-end check (not combinable)
BASE      = "bf16"             # paired comparison is made against this when it is in the same run (combine.py otherwise)
MODEL, REVISION = "allenai/OLMoE-1B-7B-0924", "6d84c48581ece794365f2b8e9cfb043c68ade9c5"  # team pin
LLAMA_TAG = "b11669"           # llama.cpp build of the team's conversion/quantization
LLAMA_CPP_PYTHON = "0.3.36"    # in-process log-probabilities checked correct with this version
G9_RAW = "https://raw.githubusercontent.com/theschoolofai/era-v5-lab/main/teams/09-evaluation/harness/run_eval_mini.py"
DRIVE_DIR = "/content/drive/MyDrive/era-v5-g12/eval_mini"   # Colab, when Drive mounts
BF16_FALLBACK_GPU_LAYERS = 12  # of 16 layers, only if bf16 does not fit on the GPU(s)
NOTEBOOK_VERSION = "ec7db60966d6"   # fingerprint of the embedded code: combine.py requires the same one
assert FORMATS and all(f in {"bf16", "Q8_0", "Q6_K", "Q5_K_M", "Q4_K_M", "Q3_K_M", "Q2_K"} for f in FORMATS), FORMATS

import os
if os.environ.get("EVAL_MINI_LOCAL_TEST"):   # dry run of these same cells on a Mac (see build_notebook.py)
    LIMIT, FORMATS, BASE, RUN_BY = 2, ["Q8_0", "Q2_K"], "Q8_0", "local-test"

In [ ]:
import datetime, glob, json, os, platform, shutil, subprocess, sys, time
LOCAL_TEST = bool(os.environ.get("EVAL_MINI_LOCAL_TEST"))
IN_KAGGLE = os.path.isdir("/kaggle/working")
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if LOCAL_TEST:
    LOCAL_OUT, WORK = os.environ.get("EVAL_MINI_OUT", "/tmp/eval_mini_local"), os.path.expanduser("~/.cache/era-v5-quant")
elif IN_KAGGLE:
    LOCAL_OUT, WORK = "/kaggle/working/eval_mini_out", "/kaggle/tmp/eval_mini_work"   # output vs big files
else:
    LOCAL_OUT, WORK = "/content/eval_mini_out", "/content/work"
OUT = LOCAL_OUT
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = DRIVE_DIR
    except Exception as e:  # e.g. not available from an editor-attached runtime
        print(f"Drive not mounted ({e!r}); output goes to {LOCAL_OUT}. DOWNLOAD THE ZIP AT THE END.")
for d in (OUT, f"{OUT}/logs", WORK):
    os.makedirs(d, exist_ok=True)
CODE = f"{WORK}/eval_mini_code"; os.makedirs(CODE, exist_ok=True)

def sh(cmd, log=None):
    """Run a shell command; stream its output (and append it to a log); stop the notebook if it fails."""
    print("$", cmd, flush=True)
    full = f"set -o pipefail; ( {cmd} ) 2>&1" + (f" | tee -a {log}" if log else "")
    rc = subprocess.run(full, shell=True, executable="/bin/bash").returncode
    if rc != 0:
        raise RuntimeError(f"command failed (exit {rc}): {cmd}" + (f"; see {log}" if log else ""))

PLATFORM = "local-test" if LOCAL_TEST else "kaggle" if IN_KAGGLE else "colab" if IN_COLAB else "other"
ENV = {"started": datetime.datetime.now().isoformat(timespec="seconds"), "platform": PLATFORM,
       "output_dir": OUT, "run_by": RUN_BY, "notebook_version": NOTEBOOK_VERSION,
       "settings": {k: globals()[k] for k in ("LIMIT", "FORMATS", "BASE", "MODEL", "REVISION",
       "LLAMA_TAG", "LLAMA_CPP_PYTHON", "BF16_FALLBACK_GPU_LAYERS")}, "python": platform.python_version(), "formats": {}}
def save_env():
    json.dump(ENV, open(f"{OUT}/environment.json", "w"), indent=1)
if shutil.which("nvidia-smi"):
    ENV["gpu"] = subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader",
                                shell=True, capture_output=True, text=True).stdout.strip()
    sh("nvidia-smi")
sh(f"df -h {WORK} | tail -1")
if not LOCAL_TEST:   # Kaggle needs Internet switched on in the notebook settings
    sh("curl -fsS -o /dev/null https://huggingface.co || (echo 'NO INTERNET: turn it on in the notebook settings' && false)")
save_env()
print("platform:", PLATFORM, "| output:", OUT, "| model files:", WORK)

In [ ]:
# the evaluation code (llamacpp_lm.py, run_mini_gguf.py, paired.py), embedded by build_notebook.py
SOURCES = {
"llamacpp_lm.py": "\"\"\"lm-evaluation-harness model class that runs a GGUF file in-process with llama-cpp-python.\n\nWhy this exists: lm_eval scores multiple-choice tasks by the log-probability of each answer. Neither server\nroute gives correct numbers for GGUF files (checked 28 Oct, llama.cpp b11669, llama-cpp-python 0.3.36):\n  - llama-server ignores `echo=True` and returns logprobs only for generated tokens, so lm_eval's\n    local-completions / gguf backends cannot score multiple choice at all;\n  - llama-cpp-python's server returns prompt logprobs SHIFTED BY ONE position (log p(\" Paris\" | \"The capital\n    of France is\") comes back as -8.58 instead of -0.49), which lm_eval would sum without any error.\nHere the model runs in-process with every position's logits kept, and the log-probability of token i is read\nfrom the distribution at position i-1. Checked: log p(\" Paris\" | ...) = -0.49, as greedy generation reports.\n\nUse: `from llamacpp_lm import LlamaCppLM; lm = LlamaCppLM(\"model.gguf\")`, then pass `model=lm` to\n`lm_eval.simple_evaluate`. Requests are scored one at a time; a cache of the previous sequence's tokens means\nthe shared question of a multiple-choice item is evaluated once, not once per answer option.\n\"\"\"\nfrom __future__ import annotations\n\nimport numpy as np\nfrom llama_cpp import Llama\nfrom lm_eval.api.model import LM\nfrom tqdm import tqdm\n\n\nclass LlamaCppLM(LM):\n    def __init__(self, model_path: str, n_ctx: int = 2048, n_gpu_layers: int = -1, max_gen_toks: int = 256,\n                 seed: int = 42, n_batch: int = 512):\n        super().__init__()\n        self.llm = Llama(model_path=model_path, n_ctx=n_ctx, n_gpu_layers=n_gpu_layers, logits_all=True,\n                         seed=seed, n_batch=n_batch, verbose=False)\n        self.n_ctx, self.max_gen_toks = n_ctx, max_gen_toks\n        meta = self.llm.metadata\n        self.add_bos = meta.get(\"tokenizer.ggml.add_bos_token\", \"false\") == \"true\"\n        bos, eos = self.llm.token_bos(), self.llm.token_eos()\n        # lm_eval's HF model uses this token as the context of an empty-context request\n        self.prefix_token = bos if self.add_bos and bos >= 0 else eos\n\n    # ------------------------------------------------------------------ helpers\n    def _tok(self, text: str, bos: bool) -> list[int]:\n        return self.llm.tokenize(text.encode(\"utf-8\"), add_bos=bos, special=False)\n\n    def _encode_pair(self, context: str, continuation: str) -> tuple[list[int], list[int]]:\n        \"\"\"Same split as lm_eval's HFLM: trailing spaces of the context move to the continuation, then the\n        continuation tokens are the tail of the jointly tokenized string (so merges across the boundary hold).\"\"\"\n        n_spaces = len(context) - len(context.rstrip())\n        if n_spaces:\n            continuation, context = context[-n_spaces:] + continuation, context[:-n_spaces]\n        if not context:\n            return [self.prefix_token], self._tok(continuation, bos=False)\n        whole = self._tok(context + continuation, bos=self.add_bos)\n        ctx = self._tok(context, bos=self.add_bos)\n        return ctx, whole[len(ctx):]\n\n    def _eval_reuse_prefix(self, tokens: list[int]) -> None:\n        \"\"\"Evaluate `tokens`, keeping the KV cache and logit rows of the longest prefix shared with the previous\n        sequence. Row j of llm.scores depends only on tokens[:j+1], so kept rows stay valid.\"\"\"\n        llm, keep = self.llm, 0\n        for a, b in zip(llm.input_ids[: llm.n_tokens], tokens):\n            if a != b:\n                break\n            keep += 1\n        keep = min(keep, len(tokens) - 1)  # always evaluate at least the last token\n        llm.n_tokens = keep                # eval() drops the KV cache from n_tokens onward\n        llm.eval(tokens[keep:])\n\n    def _score(self, context_tokens: list[int], continuation_tokens: list[int]) -> tuple[float, bool]:\n        tokens = (context_tokens + continuation_tokens)[-self.n_ctx:]   # left-truncate, as HFLM does\n        n_cont = len(continuation_tokens)\n        self._eval_reuse_prefix(tokens)\n        start = len(tokens) - n_cont\n        rows = np.asarray(self.llm.scores[start - 1: len(tokens) - 1], dtype=np.float64)  # row i-1 predicts token i\n        logsm = rows - np.logaddexp.reduce(rows, axis=1, keepdims=True)\n        targets = np.asarray(tokens[start:])\n        logprob = float(logsm[np.arange(n_cont), targets].sum())\n        greedy = bool((logsm.argmax(axis=1) == targets).all())\n        return logprob, greedy\n\n    # ------------------------------------------------------------------ lm_eval interface\n    def loglikelihood(self, requests, disable_tqdm: bool = False):\n        pairs = [self._encode_pair(*req.args) for req in requests]\n        # score requests that share a question back to back, so the prefix cache is used; return in input order\n        order = sorted(range(len(pairs)), key=lambda i: pairs[i][0])\n        out = [None] * len(pairs)\n        for i in tqdm(order, disable=disable_tqdm, desc=\"loglikelihood\"):\n            out[i] = self._score(*pairs[i])\n        return out\n\n    def loglikelihood_rolling(self, requests, disable_tqdm: bool = False):\n        raise NotImplementedError(\"not needed by the eval_mini profiles (perplexity is measured in perplexity/)\")\n\n    def generate_until(self, requests, disable_tqdm: bool = False):\n        out = []\n        for req in tqdm(requests, disable=disable_tqdm, desc=\"generate_until\"):\n            context, gen_kwargs = req.args\n            gen_kwargs = dict(gen_kwargs)\n            until = gen_kwargs.pop(\"until\", None) or []\n            until = [until] if isinstance(until, str) else list(until)\n            max_new = int(gen_kwargs.pop(\"max_gen_toks\", self.max_gen_toks))\n            if gen_kwargs.get(\"do_sample\") or float(gen_kwargs.get(\"temperature\", 0.0) or 0.0) > 0:\n                raise ValueError(f\"only greedy generation is supported, got {gen_kwargs}\")\n            prompt = self._tok(context, bos=self.add_bos)[-(self.n_ctx - max_new):]\n            self.llm.reset()  # generation manages its own cache; do not mix it with the scoring cache\n            res = self.llm.create_completion(prompt=prompt, max_tokens=max_new, temperature=0.0, top_k=1,\n                                             stop=until or None)\n            out.append(res[\"choices\"][0][\"text\"])\n        self.llm.reset()\n        return out\n",
"run_mini_gguf.py": "#!/usr/bin/env python3\n\"\"\"Run G9's eval_mini profile on one GGUF file, with per-item logs, in G9's eval_mini_v0 record format.\n\n    python run_mini_gguf.py --gguf OLMoE-1B-7B-0924-Q8_0.gguf --format Q8_0 --out runs/ \\\n        --g9-harness <era-v5-lab main>/teams/09-evaluation/harness/run_eval_mini.py\n\nThe task set, shot counts, limits, seed and record format are imported from G9's run_eval_mini.py, not copied,\nso a run here is like for like with G9's baseline. Per task it writes:\n  <out>/<format>/<task>_<k>shot.samples.jsonl   one line per item: task, doc_id, target, response, metric values\n  <out>/<format>/summary.json                   run metadata + one record per (task, metric, filter)\nA task whose records are already in summary.json is skipped, so an interrupted run resumes where it stopped.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, datetime, hashlib, importlib.util, json, platform, subprocess, sys, time\nfrom pathlib import Path\n\nHERE = Path(__file__).resolve().parent\nsys.path.insert(0, str(HERE))\n\n\ndef load_g9(path: Path):\n    spec = importlib.util.spec_from_file_location(\"run_eval_mini\", path)\n    mod = importlib.util.module_from_spec(spec)\n    sys.modules[\"run_eval_mini\"] = mod   # its dataclasses look themselves up in sys.modules while being defined\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef gpu_name() -> str:\n    try:\n        return subprocess.run([\"nvidia-smi\", \"--query-gpu=name,memory.total\", \"--format=csv,noheader\"],\n                              capture_output=True, text=True, check=True).stdout.strip().replace(\"\\n\", \" + \")\n    except (OSError, subprocess.CalledProcessError):\n        if platform.system() == \"Darwin\":\n            sc = lambda k: subprocess.run([\"sysctl\", \"-n\", k], capture_output=True, text=True).stdout.strip()\n            return f\"{sc('machdep.cpu.brand_string')} {int(sc('hw.memsize')) // 2**30}GB (Metal)\"\n        return platform.processor() or platform.machine()\n\n\ndef sample_rows(results: dict, metric_names: set[str]) -> list[dict]:\n    \"\"\"Flatten lm_eval's per-sample logs (one list per leaf task) into slim rows keyed by (task, filter, doc_id);\n    GSM8K logs each item once per answer filter (strict-match, flexible-extract).\"\"\"\n    rows = []\n    for task, samples in (results.get(\"samples\") or {}).items():\n        for s in samples:\n            resp = s.get(\"filtered_resps\")\n            rows.append({\"task\": task, \"doc_id\": s[\"doc_id\"], \"filter\": s.get(\"filter\", \"none\"), \"target\": s.get(\"target\"),\n                         \"response\": resp if not isinstance(resp, list) or len(resp) > 1 else resp[0],\n                         **{m: s[m] for m in metric_names if m in s}})\n    return rows\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--gguf\", required=True, type=Path)\n    ap.add_argument(\"--format\", required=True, help=\"label for this file, e.g. bf16, Q8_0\")\n    ap.add_argument(\"--out\", required=True, type=Path)\n    ap.add_argument(\"--g9-harness\", required=True, type=Path, help=\"path to G9's run_eval_mini.py\")\n    ap.add_argument(\"--profile\", default=\"mini\")\n    ap.add_argument(\"--tasks\", nargs=\"+\", help=\"subset of the profile's tasks\")\n    ap.add_argument(\"--limit\", type=int, help=\"override every task's limit (smoke tests only)\")\n    ap.add_argument(\"--n-ctx\", type=int, default=2048)\n    ap.add_argument(\"--n-gpu-layers\", type=int, default=-1, help=\"-1 = all layers on the GPU\")\n    ap.add_argument(\"--model\", default=\"allenai/OLMoE-1B-7B-0924\")\n    ap.add_argument(\"--revision\", default=\"6d84c48581ece794365f2b8e9cfb043c68ade9c5\")\n    args = ap.parse_args()\n\n    g9 = load_g9(args.g9_harness)\n    profile = g9.PROFILES[args.profile]\n    tasks = args.tasks or list(profile)\n    out = args.out / args.format; out.mkdir(parents=True, exist_ok=True)\n    summary_path = out / \"summary.json\"\n    summary = json.loads(summary_path.read_text()) if summary_path.exists() else {\n        \"schema_version\": g9.SCHEMA_VERSION, \"run_meta\": {}, \"records\": []}\n    done = {r[\"task\"] for r in summary[\"records\"]}\n\n    import llama_cpp, lm_eval\n    from llamacpp_lm import LlamaCppLM\n    t0 = time.perf_counter()\n    lm = LlamaCppLM(str(args.gguf), n_ctx=args.n_ctx, n_gpu_layers=args.n_gpu_layers, seed=g9.DEFAULT_SEED)\n    load_s = time.perf_counter() - t0\n    summary[\"run_meta\"] = {\n        \"model\": args.model, \"model_revision\": args.revision, \"format\": args.format,\n        \"gguf\": args.gguf.name, \"gguf_bytes\": args.gguf.stat().st_size, \"profile\": args.profile,\n        \"backend\": f\"llama-cpp-python {llama_cpp.__version__} (in-process, logits_all)\",\n        \"lm_eval\": lm_eval.__version__, \"hardware\": gpu_name(), \"n_ctx\": args.n_ctx,\n        \"n_gpu_layers\": args.n_gpu_layers, \"seed\": g9.DEFAULT_SEED, \"load_s\": round(load_s, 1),\n        \"g9_harness_sha256\": hashlib.sha256(args.g9_harness.read_bytes()).hexdigest()[:16],\n    }\n    for name in tasks:\n        if name in done:\n            print(f\"{args.format}/{name}: already done, skipped\", flush=True); continue\n        spec, run = g9.TASKS[name], profile[name]\n        limit = args.limit if args.limit is not None else run.limit\n        t = time.perf_counter()\n        results = lm_eval.simple_evaluate(\n            model=lm, tasks=[name], num_fewshot=run.num_fewshot or None, limit=limit, log_samples=True,\n            random_seed=g9.DEFAULT_SEED, numpy_random_seed=g9.DEFAULT_SEED, fewshot_random_seed=g9.DEFAULT_SEED,\n            # no torch seed: this backend does not use torch, and setting it imports torch, which is broken on\n            # Kaggle (libtorch_cuda: undefined symbol ncclCommShrink). G9's other three seeds are unchanged.\n            torch_random_seed=None)\n        secs = time.perf_counter() - t\n        context = {\"model\": args.model, \"model_revision\": args.revision, \"dtype\": args.format,\n                   \"profile\": args.profile, \"limit\": limit, \"seed\": g9.DEFAULT_SEED,\n                   \"harness_version\": lm_eval.__version__, \"run_date\": datetime.date.today().isoformat(),\n                   \"device\": \"llama-cpp-python\", \"hardware\": summary[\"run_meta\"][\"hardware\"], \"batch_size\": 1,\n                   \"wall_clock_s\": round(secs, 1)}\n        records = g9.extract_records(results, spec, run.num_fewshot, context)\n        metric_names = {r[\"metric\"] for r in records}\n        rows = sample_rows(results, metric_names)\n        with open(out / f\"{name}_{run.num_fewshot}shot.samples.jsonl\", \"w\") as fh:\n            for row in rows:\n                fh.write(json.dumps(row, ensure_ascii=False) + \"\\n\")\n        summary[\"records\"] = [r for r in summary[\"records\"] if r[\"task\"] != name] + records\n        summary_path.write_text(json.dumps(summary, indent=1))   # after every task: a crash loses one task at most\n        prim = next((r for r in records if r[\"primary\"]), records[0])\n        print(f\"{args.format}/{name}: {prim['metric']}={prim['value']:.4f} +/- {prim['stderr'] or 0:.4f} \"\n              f\"n={prim['n_items']} in {secs:.0f}s ({secs / max(prim['n_items'], 1):.2f}s/item), \"\n              f\"{len(rows)} per-item rows\", flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n",
"paired.py": "#!/usr/bin/env python3\n\"\"\"Paired comparison of each quantized format with bf16 on eval_mini, item by item.\n\n    python paired.py runs/ --base bf16 [--out paired.md]\n\nReads runs/<format>/<task>_<k>shot.samples.jsonl (written by run_mini_gguf.py) and compares every format with\nthe base on the SAME items. Comparing two accuracies with their own standard errors (as compare_v0.md does for\nmodels) needs gaps of ~2 points per 400 items; pairing removes the item-to-item variation both runs share, so a\nmuch smaller real change becomes measurable.\n\nPer task, on its primary metric (G9's choice: acc_norm for ARC-C and HellaSwag, acc for MMLU and TruthfulQA-mc2,\nexact_match with the flexible-extract filter for GSM8K):\n  0/1 metrics:      b = items right in base but wrong in the format, c = the reverse; difference = (c - b) / n,\n                    95% CI from the paired variance; p from McNemar's exact test on b and c.\n  TruthfulQA-mc2:   acc is a probability share per item; mean of the per-item differences, 95% CI, no p-value.\nMMLU's 57 subjects are pooled into one row.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse, json, math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nPRIMARY = {\"arc_challenge\": (\"acc_norm\", \"none\"), \"arc_easy\": (\"acc_norm\", \"none\"), \"hellaswag\": (\"acc_norm\", \"none\"),\n           \"mmlu\": (\"acc\", \"none\"), \"gsm8k\": (\"exact_match\", \"flexible-extract\"), \"truthfulqa_mc2\": (\"acc\", \"none\")}\nBINARY = {\"arc_challenge\", \"arc_easy\", \"hellaswag\", \"mmlu\", \"gsm8k\"}\n\n\ndef group(task: str) -> str:\n    return \"mmlu\" if task.startswith(\"mmlu\") else task\n\n\ndef load(fmt_dir: Path) -> dict[str, dict[tuple, float]]:\n    \"\"\"{group task: {(leaf task, doc_id): primary metric value}}\"\"\"\n    out = defaultdict(dict)\n    for f in sorted(fmt_dir.glob(\"*.samples.jsonl\")):\n        for line in open(f):\n            r = json.loads(line)\n            g = group(r[\"task\"])\n            if g not in PRIMARY:\n                continue\n            metric, filt = PRIMARY[g]\n            if r.get(\"filter\", \"none\") == filt and metric in r:\n                out[g][(r[\"task\"], r[\"doc_id\"])] = float(r[metric])\n    return out\n\n\ndef mcnemar_p(b: int, c: int) -> float:\n    n = b + c\n    if n == 0:\n        return 1.0\n    k = min(b, c)\n    return min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)\n\n\ndef compare(base: dict[tuple, float], other: dict[tuple, float], binary: bool) -> dict:\n    keys = sorted(base.keys() & other.keys())\n    n = len(keys)\n    if n == 0:\n        return {\"n\": 0}\n    x, y = [base[k] for k in keys], [other[k] for k in keys]\n    res = {\"n\": n, \"base\": sum(x) / n, \"fmt\": sum(y) / n, \"missing\": len(base.keys() ^ other.keys())}\n    if binary:\n        b = sum(1 for p, q in zip(x, y) if p == 1 and q == 0)\n        c = sum(1 for p, q in zip(x, y) if p == 0 and q == 1)\n        diff = (c - b) / n\n        se = math.sqrt(max((b + c) - (c - b) ** 2 / n, 0)) / n\n        res.update(diff=diff, lo=diff - 1.96 * se, hi=diff + 1.96 * se, b=b, c=c, p=mcnemar_p(b, c))\n    else:\n        d = [q - p for p, q in zip(x, y)]\n        mean = sum(d) / n\n        sd = math.sqrt(sum((v - mean) ** 2 for v in d) / (n - 1)) if n > 1 else 0.0\n        res.update(diff=mean, lo=mean - 1.96 * sd / math.sqrt(n), hi=mean + 1.96 * sd / math.sqrt(n))\n    return res\n\n\ndef main() -> None:\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"runs\", type=Path)\n    ap.add_argument(\"--base\", default=\"bf16\")\n    ap.add_argument(\"--out\", type=Path, help=\"also write the markdown table here\")\n    args = ap.parse_args()\n\n    base = load(args.runs / args.base)\n    lines = [f\"Paired against **{args.base}** on the same items. Differences in points (format minus base); \"\n             \"b = right in base, wrong in format; c = the reverse.\", \"\",\n             \"| Format | Task | n | Base | Format | Difference (95% CI) | b / c | McNemar p |\",\n             \"|---|---|---|---|---|---|---|---|\"]\n    for fmt_dir in sorted(p for p in args.runs.iterdir() if p.is_dir() and p.name != args.base):\n        other = load(fmt_dir)\n        for task in PRIMARY:\n            if task not in base or task not in other:\n                continue\n            r = compare(base[task], other[task], task in BINARY)\n            if not r[\"n\"]:\n                continue\n            pts = lambda v: f\"{100 * v:+.1f}\"\n            bc = f\"{r['b']} / {r['c']}\" if \"b\" in r else \"-\"\n            p = f\"{r['p']:.3f}\" if \"p\" in r else \"-\"\n            warn = f\" ({r['missing']} items unmatched)\" if r[\"missing\"] else \"\"\n            lines.append(f\"| {fmt_dir.name} | {task}{warn} | {r['n']} | {100 * r['base']:.1f} | {100 * r['fmt']:.1f} | \"\n                         f\"{pts(r['diff'])} ({pts(r['lo'])} to {pts(r['hi'])}) | {bc} | {p} |\")\n    text = \"\\n\".join(lines)\n    print(text)\n    if args.out:\n        args.out.write_text(text + \"\\n\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
"test_llamacpp_lm.py": "#!/usr/bin/env python3\n\"\"\"Guards llamacpp_lm.py against the shifted-logprob bug (instructor's request, 28 Oct).\n\n    python test_llamacpp_lm.py <model.gguf>        # exits non-zero if a check fails\n    EVAL_MINI_GGUF=<model.gguf> python -m pytest test_llamacpp_lm.py\n\nllama-cpp-python's server returns prompt log-probabilities one position late: log p(\" Paris\" | \"The capital of\nFrance is\") came back as -8.58, the value of the distribution AFTER \" Paris\", instead of -0.49. lm_eval sums\nsuch values without any error. The checks compare LlamaCppLM with the same model's own greedy generation, so\nthey hold for any GGUF, not only one model's numbers:\n\n  1. the score of \" Paris\" equals the log-probability greedy generation reports for that token;\n  2. it is NOT the off-by-one value (the generated token's log-probability one position later);\n  3. scoring with the prefix cache equals scoring every request from scratch;\n  4. the most likely continuation of an easy multiple-choice item is the right one.\n\"\"\"\nfrom __future__ import annotations\n\nimport os, sys\nfrom pathlib import Path\n\nimport numpy as np\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom llamacpp_lm import LlamaCppLM  # noqa: E402\n\nCONTEXT, ANSWER = \"The capital of France is\", \" Paris\"\nTOL = 0.05   # nats; GPU kernels for different batch shapes differ by ~0.004\n\n\nclass _Req:\n    def __init__(self, *args):\n        self.args = args\n\n\ndef check(gguf: str) -> list[str]:\n    lm = LlamaCppLM(gguf, n_ctx=512)\n    failures = []\n\n    # the model's own greedy next token after CONTEXT, with its log-probability, from generation\n    gen = lm.llm.create_completion(prompt=CONTEXT, max_tokens=1, temperature=0.0, top_k=1, logprobs=5)\n    top = gen[\"choices\"][0][\"logprobs\"][\"top_logprobs\"][0]\n    if ANSWER not in top:\n        return [f\"model does not rank {ANSWER!r} in its top-5 after {CONTEXT!r} ({top}); use a stronger model\"]\n    lp_gen = top[ANSWER]\n    lm.llm.reset()\n\n    lp, greedy = lm.loglikelihood([_Req(CONTEXT, ANSWER)], disable_tqdm=True)[0]\n    if abs(lp - lp_gen) > TOL:\n        failures.append(f\"1. score {lp:.3f} != generation's {lp_gen:.3f} for {ANSWER!r}\")\n\n    # the off-by-one value: the answer tokens read from the rows one position later (row i, not i-1), the\n    # server's bug; computed from the logit rows the scoring call above left behind, so it is always finite\n    ctx, cont = lm._encode_pair(CONTEXT, ANSWER)\n    rows = np.asarray(lm.llm.scores[len(ctx): len(ctx) + len(cont)], dtype=np.float64)\n    after = float((rows - np.logaddexp.reduce(rows, axis=1, keepdims=True))[np.arange(len(cont)), cont].sum())\n    if abs(lp - after) <= TOL and abs(lp_gen - after) > TOL:\n        failures.append(f\"2. score {lp:.3f} equals the off-by-one value {after:.3f}\")\n\n    reqs = [_Req(\"Question: Which gas do plants absorb from the air to make food?\\nAnswer:\", c)\n            for c in (\" Carbon dioxide\", \" Oxygen\", \" Nitrogen\", \" Helium\")]\n    lm.llm.reset()\n    cached = lm.loglikelihood(reqs, disable_tqdm=True)\n    fresh = []\n    for r in reqs:\n        lm.llm.reset()\n        fresh.append(lm.loglikelihood([r], disable_tqdm=True)[0])\n    worst = max(abs(a[0] - b[0]) for a, b in zip(cached, fresh))\n    if worst > TOL:\n        failures.append(f\"3. prefix cache changes a score by {worst:.4f}\")\n    if max(range(4), key=lambda i: cached[i][0]) != 0:\n        failures.append(f\"4. easy item answered wrong: {[round(c[0], 2) for c in cached]}\")\n\n    print(f\"{Path(gguf).name}: log p({ANSWER!r}) = {lp:.3f}, generation {lp_gen:.3f}, off-by-one value {after:.3f}; \"\n          f\"cache max diff {worst:.4f}\")\n    return failures\n\n\ndef test_logprobs_not_shifted():   # pytest entry point\n    gguf = os.environ.get(\"EVAL_MINI_GGUF\")\n    if not gguf:\n        import pytest\n        pytest.skip(\"set EVAL_MINI_GGUF to a GGUF file\")\n    assert not check(gguf)\n\n\nif __name__ == \"__main__\":\n    if len(sys.argv) != 2:\n        sys.exit(__doc__)\n    fails = check(sys.argv[1])\n    for f in fails:\n        print(\"FAIL\", f)\n    print(\"PASS\" if not fails else f\"{len(fails)} check(s) failed\")\n    sys.exit(1 if fails else 0)\n"
}
for _name, _text in SOURCES.items():
    open(f"{CODE}/{_name}", "w").write(_text)
print("wrote", sorted(SOURCES), "to", CODE)

In [ ]:
# llama-cpp-python with CUDA (compiled here, ~10-25 min). The wheel is kept where it survives a new session
# (Colab: Drive), so a later session installs it in seconds.
# GGML_CUDA_NO_VMM: Kaggle's build image has no libcuda for CMake to link (found by therahulbhati, perplexity/README.md).
#
# The evaluation runs in its OWN venv (EVAL_PY) holding only llama-cpp-python, lm_eval and datasets at G9's pins.
# lm_eval imports torch and transformers whenever it finds them, and Kaggle's preinstalled ones fail to import
# (torch: undefined symbol ncclCommShrink; transformers: a dependency version check). This venv has neither.
if LOCAL_TEST:
    LLAMA_SPEC, UV_PYTHON = f"llama-cpp-python=={LLAMA_CPP_PYTHON}", "3.12"   # uv builds it (Metal on a Mac)
else:
    UV_PYTHON = "python3"   # the platform's python, the one the wheel is built for
    sh("pip install -q uv==0.13.6", log=f"{OUT}/logs/pip.log")
    WHEELS = f"{OUT}/wheels" if OUT == DRIVE_DIR else f"{WORK}/wheels"
    wheels = glob.glob(f"{WHEELS}/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    if not wheels:
        os.makedirs(WHEELS, exist_ok=True)
        caps = sorted({c.replace(".", "") for c in subprocess.run("nvidia-smi --query-gpu=compute_cap --format=csv,noheader",
                       shell=True, capture_output=True, text=True).stdout.split()}) or ["75"]
        ENV["cuda_arch"] = caps; save_env()
        # -v makes the build print ninja's [done/total] counter; the full output goes to the log, the counter here
        build_log = f"{OUT}/logs/build_llama_cpp_python.log"
        print("compiling llama-cpp-python with CUDA: ~15-30 min; progress every ~20 kernels below")
        sh(f'CMAKE_ARGS="-DGGML_CUDA=on -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES={";".join(caps)}" '
           f'pip wheel -v --no-deps --no-cache-dir -w {WHEELS} llama-cpp-python=={LLAMA_CPP_PYTHON} 2>&1 '
           f'| tee {build_log} | grep --line-buffered -E "^ *[[][0-9]+/[0-9]+[]]" | awk "NR % 20 == 1 {{print; fflush()}}" ; '
           f'ls {WHEELS}/llama_cpp_python-*.whl >/dev/null || (tail -40 {build_log}; false)')
        wheels = glob.glob(f"{WHEELS}/llama_cpp_python-{LLAMA_CPP_PYTHON}-*.whl")
    LLAMA_SPEC = wheels[0]

EVAL_VENV = f"{WORK}/eval-venv"
EVAL_PY = f"env -u PYTHONPATH {EVAL_VENV}/bin/python"   # also without Kaggle's PYTHONPATH sitecustomize hook
if not os.path.exists(f"{EVAL_VENV}/.ready"):
    sh(f"rm -rf {EVAL_VENV} && uv venv -q --python {UV_PYTHON} {EVAL_VENV} && uv pip install -q --python {EVAL_VENV}/bin/python "
       f"{LLAMA_SPEC} lm_eval==0.4.13 datasets==5.1.0 && touch {EVAL_VENV}/.ready", log=f"{OUT}/logs/eval_venv.log")
probe = f"{CODE}/probe_env.py"
open(probe, "w").write(os.linesep.join([
    "import importlib.util, json, llama_cpp, lm_eval, datasets, platform",
    "print(json.dumps({'python': platform.python_version(), 'llama_cpp_python': llama_cpp.__version__,",
    "  'lm_eval': lm_eval.__version__, 'datasets': datasets.__version__,",
    "  'gpu_offload': bool(llama_cpp.llama_supports_gpu_offload()),",
    "  'torch_visible': importlib.util.find_spec('torch') is not None,",
    "  'transformers_visible': importlib.util.find_spec('transformers') is not None}))", ""]))
res = subprocess.run(f"{EVAL_PY} {probe}", shell=True, capture_output=True, text=True)
if res.returncode != 0:
    raise RuntimeError("the evaluation venv does not work: " + res.stderr[-3000:])
ENV["packages"] = json.loads(res.stdout.strip().splitlines()[-1])
if not LOCAL_TEST and not ENV["packages"]["gpu_offload"]:
    raise RuntimeError("llama-cpp-python has no GPU offload: delete the wheels folder and rerun to rebuild it with CUDA")
if ENV["packages"]["torch_visible"] or ENV["packages"]["transformers_visible"]:
    raise RuntimeError(f"torch/transformers visible in the evaluation venv: {ENV['packages']}")
save_env(); print(ENV["packages"])

In [ ]:
# G9's runner (profile, task list, record format) from upstream main; its sha256 goes into every summary
sh(f"curl -fsSL {G9_RAW} -o {CODE}/run_eval_mini.py")
sys.path.insert(0, CODE)

In [ ]:
# bf16 GGUF: the llama.cpp converter at the team's build, pinned checkpoint; llama-quantize built CPU-only (~5 min)
NAME = MODEL.split("/")[-1]
BF16 = f"{WORK}/{NAME}-bf16.gguf"
QUANTIZE = shutil.which("llama-quantize") if LOCAL_TEST else f"{WORK}/llama.cpp/build/bin/llama-quantize"
if not LOCAL_TEST and not os.path.exists(QUANTIZE):
    if not os.path.exists(f"{WORK}/llama.cpp"):
        sh(f"git clone -q --depth 1 --branch {LLAMA_TAG} https://github.com/ggml-org/llama.cpp {WORK}/llama.cpp")
    sh(f"cmake -S {WORK}/llama.cpp -B {WORK}/llama.cpp/build -DLLAMA_CURL=OFF -DGGML_CUDA=OFF "
       f"&& cmake --build {WORK}/llama.cpp/build -j --target llama-quantize", log=f"{OUT}/logs/build_llama_quantize.log")
if not os.path.exists(BF16):
    # the converter's own requirements (pinned torch, numpy, ...) go into a separate venv, so they cannot replace
    # packages this notebook has already imported
    CONV = f"{WORK}/convert-venv"
    if not os.path.exists(f"{CONV}/.ready"):   # uv: Kaggle's python has no ensurepip, so plain venv fails
        sh(f"rm -rf {CONV} && pip install -q uv==0.13.6 && uv venv -q --python python3 {CONV} "
           f"&& uv pip install -q --python {CONV}/bin/python --index-strategy unsafe-best-match "
           f"-r {WORK}/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt {WORK}/llama.cpp/gguf-py "
           f"&& touch {CONV}/.ready", log=f"{OUT}/logs/convert_venv.log")
    from huggingface_hub import snapshot_download
    hf = snapshot_download(MODEL, revision=REVISION, local_dir=f"{WORK}/hf",
                           allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    # env -u PYTHONPATH: Kaggle injects a sitecustomize hook through PYTHONPATH that needs packages this venv lacks
    sh(f"env -u PYTHONPATH {CONV}/bin/python {WORK}/llama.cpp/convert_hf_to_gguf.py {hf} --outtype bf16 --outfile {BF16}",
       log=f"{OUT}/logs/convert.log")
    shutil.rmtree(f"{WORK}/hf", ignore_errors=True)
ENV["bf16_gguf_bytes"] = os.path.getsize(BF16); save_env()
print(BF16, round(os.path.getsize(BF16) / 1e9, 2), "GB")

In [ ]:
# every format: quantize (same rule as scripts/quantize_eval.py: output layer Q8_0 below 8 bits), run G9's profile,
# save after each task. A format whose five tasks are all in its summary.json is skipped.
TASKS = ["arc_challenge", "hellaswag", "mmlu", "gsm8k", "truthfulqa_mc2"]
RUNS = f"{OUT}/runs_" + "-".join(FORMATS) + (f"_limit{LIMIT}" if LIMIT else "")   # unique per member; tests apart

def finished(fmt):
    p = f"{RUNS}/{fmt}/summary.json"
    return os.path.exists(p) and {r["task"] for r in json.load(open(p))["records"]} >= set(TASKS)

def run_format(fmt, gguf, gpu_layers):
    sh(f"cd {CODE} && {EVAL_PY} run_mini_gguf.py --gguf {gguf} --format {fmt} --out {RUNS} "
       f"--g9-harness {CODE}/run_eval_mini.py --n-gpu-layers {gpu_layers}" + (f" --limit {LIMIT}" if LIMIT else ""),
       log=f"{OUT}/logs/run_{fmt}.log")

for fmt in FORMATS:
    if finished(fmt):
        print(f"{fmt}: all tasks done, skipped"); continue
    t0 = time.time()
    gguf = BF16 if fmt == "bf16" else f"{WORK}/{NAME}-{fmt}.gguf"
    made_here = False
    if not os.path.exists(gguf):
        extra = "" if fmt == "Q8_0" else "--output-tensor-type q8_0"
        sh(f"{QUANTIZE} {extra} {BF16} {gguf} {fmt}", log=f"{OUT}/logs/quantize_{fmt}.log"); made_here = True
    # the shifted-logprob guard (instructor, 28 Oct): this file's scores must equal its own greedy generation's
    sh(f"cd {CODE} && {EVAL_PY} test_llamacpp_lm.py {gguf}", log=f"{OUT}/logs/test_llamacpp_lm_{fmt}.log")
    gpu_layers = -1
    try:
        run_format(fmt, gguf, gpu_layers)
    except RuntimeError:
        # retry with fewer GPU layers only for bf16 and only when the failure was GPU memory; anything else is a bug
        tail = open(f"{OUT}/logs/run_{fmt}.log").read()[-20000:].lower()
        if fmt != "bf16" or not any(k in tail for k in ("out of memory", "failed to allocate", "cudamalloc")):
            raise
        gpu_layers = BF16_FALLBACK_GPU_LAYERS
        print(f"bf16 ran out of GPU memory with all layers on the GPU; retrying with {gpu_layers} GPU layers")
        run_format(fmt, gguf, gpu_layers)
    ENV["formats"][fmt] = {"gpu_layers": gpu_layers, "gguf_bytes": os.path.getsize(gguf),
                           "wall_clock_s": round(time.time() - t0), "finished": datetime.datetime.now().isoformat(timespec="seconds")}
    save_env()
    if made_here and fmt != "bf16" and not LOCAL_TEST:
        os.remove(gguf)   # keep the local disk small

In [ ]:
# results: G9-primary scores per format, the paired comparison with the base, a zip of everything
import pandas as pd
rows = []
for p in sorted(glob.glob(f"{RUNS}/*/summary.json")):
    s = json.load(open(p))
    for r in s["records"]:
        if r["primary"]:
            rows.append({"format": s["run_meta"]["format"], "task": r["task"], "points": round(100 * r["value"], 1),
                         "stderr": round(100 * (r["stderr"] or 0), 1), "n": r["n_items"], "seconds": r["wall_clock_s"]})
long = pd.DataFrame(rows)
long.to_csv(f"{RUNS}/scores_long.csv", index=False)
scores = long.pivot_table(index="format", columns="task", values="points").reindex([f for f in FORMATS if f in set(long.format)])
scores.to_csv(f"{RUNS}/scores.csv")
print(scores.to_string())
if os.path.exists(f"{RUNS}/{BASE}/summary.json"):
    sh(f"{EVAL_PY} {CODE}/paired.py {RUNS} --base {BASE} --out {RUNS}/paired.md")
else:
    print(f"{BASE} is not in this run: the paired comparison is made when the runs are combined (combine.py)")
ENV["finished"] = datetime.datetime.now().isoformat(timespec="seconds"); save_env()
shutil.copy(f"{OUT}/environment.json", f"{RUNS}/environment.json")
shutil.copytree(f"{OUT}/logs", f"{RUNS}/logs", dirs_exist_ok=True)
zip_path = shutil.make_archive(RUNS, "zip", RUNS)
print("\nEverything is in", RUNS, "and", zip_path)
n_files = sum(len(f) for _, _, f in os.walk(RUNS)); print(n_files, "files:", sorted(os.listdir(RUNS)))

In [ ]:
# Colab without Drive only: the zip as base64 text in this cell's output, so it survives the runtime (e.g. a
# Colab kernel attached from VS Code: files stay on Google's VM, cell outputs are saved with the notebook).
# Locally: python extract_results.py eval_mini.ipynb  -> writes the zip next to the notebook.
import base64
if IN_COLAB and OUT != DRIVE_DIR:
    blob = base64.b64encode(open(zip_path, "rb").read()).decode()
    print(f"EVAL_MINI_ZIP_BEGIN {os.path.basename(zip_path)} {len(blob)}")
    for i in range(0, len(blob), 4000):
        print(blob[i:i + 4000])
    print("EVAL_MINI_ZIP_END")
else:
    print("results:", zip_path, "(Kaggle: the version's Output tab; Colab with Drive: the Drive folder)")

## Send back

**Kaggle:** download `eval_mini_out/runs*.zip` from the version's Output tab. **Colab with Drive:** it is in the
Drive folder. **Colab without Drive:** save the notebook, then locally `python extract_results.py eval_mini.ipynb`.
The zip is named after your formats (e.g. `runs_Q8_0-Q6_K.zip`); send it to the person combining the results.
It holds per format a `summary.json`
(G9's record format) and one `*.samples.jsonl` per task; `scores.csv`, `scores_long.csv`, `paired.md`,
`environment.json` (GPU, versions, settings, per-format GPU layers and time) and `logs/`. From Drive, or if Drive
did not mount, download it from the runtime before the session ends.